## Regression-Based Harmonisation

As a simple baseline, we can use regression to remove site-related mean shifts from the data.

For each feature, we fit a model like:

$$
y = \beta_0 + \beta_1 \text{Age} + \beta_2 \text{Timepoint} + \gamma \text{Site} + \epsilon
$$

Here:

- the **site term** captures additive scanner/batch effects,
- age and timepoint are kept as biological covariates,
- the residual part represents variation not explained by site.

# Regression-Based Harmonisation Workflow

In this section, we will use a simple regression-based approach to reduce site-related variability in the simulated data.

The goal is to demonstrate:
- what regression-based harmonisation can correct,
- and why more advanced methods such as ComBat-like approaches may still be needed.

---

## Steps

### 1. Simulate longitudinal multi-site data

We first generate synthetic imaging data containing:

- repeated measurements,
- biological variation,
- covariate effects (e.g., Age),
- additive site effects,
- and multiplicative site effects.

This creates a realistic harmonisation scenario where scanner/site effects are intentionally introduced.

---

### 2. Visualize site-related effects

Before harmonisation, we inspect the distributions across sites using boxplots.

This helps identify:
- additive effects → shifts in mean values,
- multiplicative effects → differences in spread/variance.

---

### 3. Apply regression-based harmonisation

We fit regression models that include:
- biological covariates,
- timepoint information,
- and site/scanner terms.

The estimated site effects are then removed from the data.

This approach primarily targets additive batch effects.

---

### 4. Visualize the harmonized data

We compare the distributions before and after correction to assess:
- reduction in site-related mean shifts,
- preservation of biological variability,
- and any remaining variance differences.

---

### 5. Evaluate remaining batch effects

Finally, we re-run:
- additive effect tests,
- multiplicative effect tests,
- and reliability metrics.

This allows us to evaluate how much scanner/site variability remains after harmonisation.

---

## Key Concept

Regression-based harmonisation is useful for correcting systematic mean shifts across sites.

However, if scanners also differ in variability or scale, regression alone may not be sufficient.

This motivates the use of ComBat-like harmonisation methods, which aim to model and remove both:
- additive effects,
- and multiplicative effects.

In [ ]:
from block2_utils.HarmonisationEvaluation_functions import (
    simulate_longitudinal_batch_data_mixed,
)
from block2_utils.HarmonisationEvaluation_plots import (
    plot_additive_multiplicative_effects,
)

# Simulate 100 subjects, 3 timepoints, 2 sites, 4 features
n_subjects=100
n_timepoints=3
n_sites=2
n_features=4
additive_shift = {"Site_B": {"Feature_1": 800, "Feature_3": -150}}
multiplicative_scale = {"Site_B": {"Feature_2": 30.0}, "Site_A": {"Feature_4": 10}}

df = simulate_longitudinal_batch_data_mixed(
    n_subjects=n_subjects,
    n_timepoints=n_timepoints,
    n_sites=n_sites,
    n_features=n_features,
    additive_shift=additive_shift,
    multiplicative_scale=multiplicative_scale,
    seed=1,
)

print(df.head(10))
feature_cols = [f"Feature_{i}" for i in range(1, n_features + 1)]

# Visualise additive and multiplicative effects
plot_additive_multiplicative_effects(df, feature_cols=feature_cols, batch_col="Site")


In [ ]:
from block2_utils.HarmonisationEvaluation_functions import regression_harmonize_site
from block2_utils.HarmonisationEvaluation_plots import plot_before_after_by_site


feature_cols = [f"Feature_{i}" for i in range(1, n_features + 1)]

df_harm, model_summary = regression_harmonize_site(
    df, feature_cols=feature_cols, site_col="Site", covariates=("Age", "Timepoint")
)
print("=="*40)
print("Model Summary")
print("=="*40)
print(model_summary)
print("=="*40)
print(df_harm.head())
print("=="*40)
plot_before_after_by_site(df_harm, feature_cols)

## Interpreting the Regression Output

The regression summary table provides feature-wise information about the relationship between the imaging measurements and site effects.

Typical columns include:

- **Feature**  
  The imaging feature/region being analysed.

- **Formula**  
  The regression model used for harmonisation.

- **Site_pvalue**  
  Statistical significance of the site/scanner term.

  - small p-values suggest strong site-related mean shifts
  - non-significant values suggest weaker additive batch effects

- **R2**  
  Proportion of variance explained by the regression model.

Higher values indicate that the model explains a larger portion of variability in the feature.

### Expected observations

Before harmonisation:
- regions with simulated additive effects should often show significant site terms.

After regression-based harmonisation:
- site-related mean differences should reduce,
- and visual separation across sites should become smaller.

However, variance differences may still remain for regions with multiplicative batch effects.

## Exercise: Regression-Based Harmonisation

In this exercise, simulate longitudinal multi-site imaging data containing:

- additive batch effects,
- multiplicative batch effects,
- and biological covariates (e.g., Age).

Then:

1. Visualize the site-related effects before harmonisation.
2. Apply regression-based harmonisation.
3. Visualize the harmonized data.
4. Re-run additive and multiplicative effect tests.

### Questions

- Which site effects were reduced after regression harmonisation?
- Which effects still remained?
- Why might regression be insufficient for multiplicative batch effects?
- How could ComBat-like methods improve upon this?

### Solution

You can check the solution code in [here](https://github.com/N-Nieto/OHBM2026_Educational_course_harmonization/tree/main/solutions/block02).
